# Autoencoder IDS Model — UAVCAN Intrusion Dataset
**Source:** Korea University HCRL Lab (IEEE DataPort)  
**Attack Types:** Normal, Attack (binary classification)  
**Features:** 10 CAN bus features (CAN_ID, DLC, byte0-byte7)  
**Approach:** Train on Normal only, flag high reconstruction error as attack  
**Output:** ae_model.keras, ae_results.json

In [1]:
# ============================================================
# Cell 2: Load data
# WHAT: Load preprocessed dataset files for Autoencoder model
#
# WHY:  Autoencoder is unsupervised anomaly detection —
#       it trains ONLY on normal (benign) traffic.
#       It learns to reconstruct normal patterns.
#       High reconstruction error = attack detected.
#       label_classes needed to identify benign label number.
#
# HOW:  Step 1: load X_train, X_test, y_train, y_test
#       Step 2: load label_classes to find benign number
#       Step 3: filter X_train_normal = benign rows only
#       Step 4: AE trains on X_train_normal only
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import time
import json

save_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAVCAN/processed/"

X_train = np.load(save_path + "X_train.npy")
X_test  = np.load(save_path + "X_test.npy")
y_train = pd.read_csv(save_path + "y_train.csv").squeeze()
y_test  = pd.read_csv(save_path + "y_test.csv").squeeze()
label_classes = pd.read_csv(save_path + "label_classes.csv").squeeze().tolist()
feature_names = pd.read_csv(save_path + "feature_names.csv").squeeze().tolist()

# benign label number — check label_classes to confirm
print(f"Label classes: {label_classes}")
# UAV_Attack: benign=2, ISOT: Benign=0, UAVCAN: Normal=1

# filter benign rows only for training
# change the number based on dataset:
# UAV_Attack → 2, ISOT → 0, UAVCAN → 1
BENIGN_LABEL = 1 

X_train_normal = X_train[y_train == BENIGN_LABEL]

print(f"X_train total:  {X_train.shape}")
print(f"X_train normal: {X_train_normal.shape}")
print(f"X_test:         {X_test.shape}")
print(f"Input features: {X_train.shape[1]}")
print("Data loaded!")

2026-05-15 15:53:57.752459: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Label classes: ['Attack', 'Normal']
X_train total:  (1422771, 10)
X_train normal: (893210, 10)
X_test:         (609759, 10)
Input features: 10
Data loaded!


In [2]:
# ============================================================
# Cell 3 Build Autoencoder:
# WHAT: Build Autoencoder architecture
#
# WHY:  Autoencoder learns to compress then reconstruct
#       normal traffic. When it sees attack traffic,
#       reconstruction error is high — flagged as anomaly.
#       Unlike RF/CNN, no attack labels needed for training.
#       This simulates real-world where attacks are unknown.
#
# HOW:  Encoder: input → 32 → 16 (compress)
#       Decoder: 16 → 32 → input (reconstruct)
#       Loss = MSE (mean squared error between input and output)
#       Low MSE = normal traffic (model reconstructs well)
#       High MSE = attack traffic (model struggles to reconstruct)
# ============================================================

print("Building Autoencoder...")

input_dim = X_train.shape[1]  # auto-detect feature count

# Encoder
input_layer = keras.Input(shape=(input_dim,))
encoded = keras.layers.Dense(32, activation='relu')(input_layer)
encoded = keras.layers.Dense(16, activation='relu')(encoded)

# Decoder
decoded = keras.layers.Dense(32, activation='relu')(encoded)
decoded = keras.layers.Dense(input_dim, activation='linear')(decoded)

# Autoencoder model
autoencoder = keras.Model(input_layer, decoded)
autoencoder.compile(optimizer='adam', loss='mse')

autoencoder.summary()
print("Autoencoder built!")

Building Autoencoder...


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 10)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │           352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │           330 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,754 (6.85 KB)

 Trainable params: 1,754 (6.85 KB)

 Non-trainable params: 0 (0.00 B)

Autoencoder built!


In [3]:
# ============================================================
# Cell 4 Train:
# WHAT: Train Autoencoder on NORMAL traffic only
#
# WHY:  Training on benign only forces the model to learn
#       what normal looks like. It cannot learn attack patterns.
#       When attack traffic appears, reconstruction fails
#       producing high MSE — our anomaly signal.
#       Training time recorded as SE metric.
#
# HOW:  input = output (X_train_normal → X_train_normal)
#       epochs=10 = 10 full passes through normal data
#       batch_size=512 = process 512 samples at a time
#       validation_split=0.1 = monitor overfitting
# ============================================================

print("Training Autoencoder on normal traffic only...")
start_time = time.time()

history = autoencoder.fit(
    X_train_normal, X_train_normal,  # input = output
    epochs=10,
    batch_size=512,
    validation_split=0.1,
    verbose=1
)

ae_train_time = round(time.time() - start_time, 2)
print(f"Training complete! Time: {ae_train_time}s")

Training Autoencoder on normal traffic only...
Epoch 1/10
1571/1571 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.1928 - val_loss: 0.0018
Epoch 2/10
1571/1571 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0010 - val_loss: 3.9601e-04
Epoch 3/10
1571/1571 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 3.0041e-04 - val_loss: 2.1823e-04
Epoch 4/10
1571/1571 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 1.4777e-04 - val_loss: 1.0480e-04
Epoch 5/10
1571/1571 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 9.1397e-05 - val_loss: 1.0599e-04
Epoch 6/10
1571/1571 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 6.9737e-05 - val_loss: 5.8302e-05
Epoch 7/10
1571/1571 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 5.5299e-05 - val_loss: 6.5759e-05
Epoch 8/10
1571/1571 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 4.4952e-05 - val_loss: 4.4608e-05
Epoch 9/10
1571/1571 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 3.9090e-05 - val_loss: 3.6182e-05
Epoch 10/10
1571/1571 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 3.5319e-05 - val_loss: 3.7549e-05
Trai

In [4]:
# ============================================================
# Cell 5 Calculate reconstruction error:
# WHAT: Calculate reconstruction error (MSE) for all test samples
#
# WHY:  MSE measures how well AE reconstructs each sample.
#       Normal traffic → low MSE (AE learned this pattern)
#       Attack traffic → high MSE (AE never saw this pattern)
#       We use 95th percentile of normal MSE as threshold.
#       Above threshold = predicted attack.
#
# HOW:  Step 1: predict() → reconstruct all test samples
#       Step 2: MSE = mean((original - reconstructed)^2)
#       Step 3: get normal test MSE only
#       Step 4: threshold = 95th percentile of normal MSE
# ============================================================

print("Calculating reconstruction errors...")
start_time = time.time()

X_pred = autoencoder.predict(X_test, batch_size=512, verbose=1)
mse = np.mean(np.power(X_test - X_pred, 2), axis=1)

ae_pred_time = round(time.time() - start_time, 2)

print(f"MSE shape: {mse.shape}")
print(f"Mean MSE:  {np.mean(mse):.4f}")
print(f"Max MSE:   {np.max(mse):.4f}")
print(f"Min MSE:   {np.min(mse):.4f}")
print("Reconstruction errors calculated!")

Calculating reconstruction errors...
1191/1191 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
MSE shape: (609759,)
Mean MSE:  0.0143
Max MSE:   0.6879
Min MSE:   0.0000
Reconstruction errors calculated!


In [5]:
# ============================================================
# Cell 6 Find threshold and evaluate:
# WHAT: Set anomaly threshold and evaluate model performance
#
# WHY:  95th percentile of normal MSE = threshold
#       meaning 95% of normal traffic is below threshold
#       Above threshold = predicted attack
#       This converts AE reconstruction error into binary prediction
#       We compare against true binary labels (benign vs attack)
#
# HOW:  Step 1: get MSE for normal test samples only
#       Step 2: threshold = 95th percentile
#       Step 3: y_pred_binary = 1 if MSE > threshold else 0
#       Step 4: y_true_binary = 1 if not benign else 0
#       Step 5: calculate Accuracy, Precision, Recall, F1
# ============================================================

y_test_reset = y_test.reset_index(drop=True)

# get normal MSE for threshold calculation
normal_mse = mse[y_test_reset == BENIGN_LABEL]
threshold = np.percentile(normal_mse, 95)
print(f"Threshold (95th percentile): {threshold:.4f}")

# binary predictions
y_pred_binary = (mse > threshold).astype(int)   # 1=attack, 0=normal
y_true_binary = (y_test_reset != BENIGN_LABEL).astype(int)  # 1=attack, 0=normal

# calculate metrics
acc = accuracy_score(y_true_binary, y_pred_binary)
p   = precision_score(y_true_binary, y_pred_binary)
r   = recall_score(y_true_binary, y_pred_binary)
f1  = f1_score(y_true_binary, y_pred_binary)

print(f"\n=== Autoencoder Results ===")
print(f"Accuracy:  {acc:.4f} ({acc*100:.2f}%)")
print(f"Precision: {p:.4f} ({p*100:.2f}%)")
print(f"Recall:    {r:.4f} ({r*100:.2f}%)")
print(f"F1 Score:  {f1:.4f} ({f1*100:.2f}%)")
print(f"Threshold: {threshold:.4f}")
print(f"Training time:   {ae_train_time}s")
print(f"Prediction time: {ae_pred_time}s")

Threshold (95th percentile): 0.0001

=== Autoencoder Results ===
Accuracy:  0.7176 (71.76%)
Precision: 0.7942 (79.42%)
Recall:    0.3255 (32.55%)
F1 Score:  0.4618 (46.18%)
Threshold: 0.0001
Training time:   59.52s
Prediction time: 4.88s


In [6]:
# ============================================================
# Cell 7 Save:
# WHAT: Save trained AE model and results to disk
#
# WHY:  Saving model means no need to retrain for SHAP/LIME/PI
#       JSON results used for paper tables and comparisons
#       Same save format across all datasets for consistency
#
# HOW:  Step 1: save model as .keras file
#       Step 2: save metrics as JSON file
# ============================================================

autoencoder.save(save_path + "ae_model.keras")

ae_results = {
    "model": "Autoencoder",
    "dataset": "UAVCAN", 
    "accuracy": round(acc, 4),
    "precision": round(p, 4),
    "recall": round(r, 4),
    "f1_score": round(f1, 4),
    "threshold": round(float(threshold), 6),
    "training_time_seconds": ae_train_time,
    "prediction_time_seconds": ae_pred_time,
    "epochs": 10,
    "batch_size": 512
}

with open(save_path + "ae_results.json", "w") as f:
    json.dump(ae_results, f, indent=4)

print("Model saved: ae_model.keras")
print("Results saved: ae_results.json")
print(f"\nResults summary:")
for k, v in ae_results.items():
    print(f"  {k}: {v}")

Model saved: ae_model.keras
Results saved: ae_results.json

Results summary:
  model: Autoencoder
  dataset: UAVCAN
  accuracy: 0.7176
  precision: 0.7942
  recall: 0.3255
  f1_score: 0.4618
  threshold: 8.8e-05
  training_time_seconds: 59.52
  prediction_time_seconds: 4.88
  epochs: 10
  batch_size: 512


## Autoencoder Results Summary — UAVCAN Intrusion Dataset

| Metric | Value |
|--------|-------|
| Dataset | UAVCAN Intrusion Dataset (Korea University) |
| Model | Autoencoder (10 epochs, batch=512) |
| Train set | 893,210 normal rows only |
| Test set | 609,759 rows |
| Features | 10 CAN bus features |
| Threshold | 0.0001 (95th percentile of normal MSE) |
| Accuracy | 71.76% |
| Precision | 79.42% |
| Recall | 32.55% |
| F1 Score | 46.18% |
| Training time | 59.52s |
| Prediction time | 4.88s |

### Key Finding
AE achieved only **46.18% F1** — worst performance across all datasets.
Very low Recall (32.55%) means AE misses 67% of CAN bus attacks.
Root cause: only 10 features — CAN bus Normal and Attack traffic
have very similar reconstruction patterns making anomaly
detection unreliable. The extremely low threshold (0.0001)
confirms Normal and Attack patterns are nearly indistinguishable
to the Autoencoder. Supervised models (RF=90.85%, CNN=88.59%)
significantly outperform AE on this dataset.